# M4 bridge — from CG geometry to the ODE, and to the bench

**BruSafe / CAPSID · iGEM 2026 · Module 4 — Assembly**

This notebook is self-contained: the first code cell writes the simulation source to disk, so nothing has to be downloaded. It runs on Colab, in Jupyter, or on a laptop with `numpy`, `scipy` and `matplotlib`.

The theory and the bench protocols (§11) are described in `M4_theory_compendium.pdf`, a separate document that is not included in this repository.

**What this notebook does.** The contact ladder of L1 is an assumption, and L3 can measure it. This notebook closes the loop:

1. take the bond census of the CG clusters,
2. fit a one-parameter cap law $b(j) = 2.5(j-1) - a\left[\sqrt{j(1-j/N)} - s_1\right]$,
3. differentiate it into a contact ladder $c(j)$ and rebuild the ODE with it,
4. push that ODE into two **benchtop observables**, plate-reader turbidity and a TEM size distribution,
5. and carry out the *corrective* step: invert a measured (optimum position, optimum height) pair into the two energies, with the CG geometry held fixed.

**Step 5 is the step that matters once measurements are available:** the geometry comes from simulation and the energetics from the bench.

In [ ]:
# Colab / fresh machine: uncomment
# !pip -q install numpy scipy matplotlib
import numpy as np, matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

In [ ]:
%%writefile brusafe_cg_assembly.py
"""
BruSafe Module 4 - Coarse-Grained Capsid Assembly (patchy-particle Metropolis MC)
==================================================================================
A spatial, orientation-resolved coarse-grained model of MS2 coat-protein-dimer
self-assembly. Each subunit is one rigid patchy particle (the coat-protein dimer,
the assembly unit established in Module 1). Binding is orientation-dependent
(Kern-Frenkel patches) with a curvature-encoding geometry so that growing sheets
close into a shell. Pure numpy + NVT Metropolis Monte Carlo - no GPU, no MD engine.

This is the SPATIAL companion to the mean-field nucleation-elongation ODEs of
Module 4. It independently reproduces the kinetic trap: assembly QUALITY (fraction
of well-coordinated subunits) is non-monotonic in subunit concentration, peaking
at intermediate packing fraction and collapsing when crowding spawns competing
nuclei that cannot anneal. Cross-scale agreement between two models built on
different assumptions (well-mixed rate equations vs explicit diffusing bodies) is
the validation argument.

Physics / parameter provenance
------------------------------
- Dimer = assembly subunit:            Ni et al. 1995 (10.1016/s0969-2126(01)00156-3)
- Patchy-particle / Kern-Frenkel CG:   Perlmutter & Hagan 2015 (10.1146/annurev-physchem-040214-121637)
- Nucleation-elongation, kinetic trap: Endres & Zlotnick 2002; Hagan 2014
- Bond free energy scale (eps ~ kT):   Ceres & Zlotnick 2002 (dimer-dimer ~ -4 kcal/mol -> ~7 kT at 310 K)

Reduced units: subunit diameter SIGMA = 1 (~5 nm for a dimer); energies in kT.
"""
import numpy as np
from collections import Counter

# ---- geometry constants -----------------------------------------------------
SIGMA   = 1.0                       # subunit diameter (reduced units)
R_WELL  = 1.20 * SIGMA              # attractive-well outer edge
PATCH_HALFWIDTH_DEG = 28.0          # Kern-Frenkel angular tolerance of an interface
COS_TOL = np.cos(np.radians(PATCH_HALFWIDTH_DEG))

# ---- subunit patch geometry -------------------------------------------------
def make_patches(kind="curved5"):
    """Body-frame unit vectors for the binding patches, tilted onto a cone of
    half-angle (90 - wedge) so each subunit prefers an out-of-plane bond angle;
    that built-in curvature makes flat sheets close into a finite shell."""
    if kind == "curved5":   n, wedge = 5, 20.0
    elif kind == "curved6": n, wedge = 6, 12.0
    else:                   n, wedge = int(kind[-1]), 18.0
    tilt = np.radians(90.0 - wedge)
    az = np.linspace(0, 2*np.pi, n, endpoint=False)
    P = np.stack([np.sin(tilt)*np.cos(az), np.sin(tilt)*np.sin(az),
                  np.cos(tilt)*np.ones(n)], axis=1)
    return P / np.linalg.norm(P, axis=1, keepdims=True)

# ---- rotation helpers -------------------------------------------------------
def rand_rot(rng):
    u1,u2,u3 = rng.random(3)
    q = np.array([np.sqrt(1-u1)*np.sin(2*np.pi*u2), np.sqrt(1-u1)*np.cos(2*np.pi*u2),
                  np.sqrt(u1)*np.sin(2*np.pi*u3),  np.sqrt(u1)*np.cos(2*np.pi*u3)])
    w,x,y,z = q
    return np.array([[1-2*(y*y+z*z),2*(x*y-w*z),2*(x*z+w*y)],
                     [2*(x*y+w*z),1-2*(x*x+z*z),2*(y*z-w*x)],
                     [2*(x*z-w*y),2*(y*z+w*x),1-2*(x*x+y*y)]])

def small_rot(rng, sigma_ang):
    ax = rng.normal(size=3); ax/=np.linalg.norm(ax)
    th = rng.normal()*sigma_ang
    c,s = np.cos(th),np.sin(th); x,y,z = ax; C=1-c
    return np.array([[c+x*x*C,x*y*C-z*s,x*z*C+y*s],
                     [y*x*C+z*s,c+y*y*C,y*z*C-x*s],
                     [z*x*C-y*s,z*y*C+x*s,c+z*z*C]])

# ---- pair energy (hard core + Kern-Frenkel patch + curvature reward) --------
def pair_energy(dr, r, Ri, Rj, P, eps):
    if r < SIGMA:   return np.inf          # excluded volume
    if r > R_WELL:  return 0.0
    dhat = dr / r
    pi = Ri @ P.T; pj = Rj @ P.T
    ci = pi.T @ dhat; cj = pj.T @ (-dhat)
    if ci.max() > COS_TOL and cj.max() > COS_TOL:
        ang  = pi[:, ci.argmax()] @ (-pj[:, cj.argmax()])   # correct dihedral -> anti-aligned
        curv = 0.5*(1+ang)
        return -eps*(0.5 + 0.5*curv)
    return 0.0

# ---- cell list --------------------------------------------------------------
def build_cells(pos, L, rc):
    n = max(1, int(L/rc)); cs = L/n
    idx = np.floor(pos/cs).astype(int) % n
    cells = {}
    for i,(a,b,c) in enumerate(idx):
        cells.setdefault((a,b,c),[]).append(i)
    return cells, n, cs

def neighbors(i, pos, cells, n, cs, L):
    a,b,c = (pos[i]//cs).astype(int) % n
    out=[]
    for da in (-1,0,1):
        for db in (-1,0,1):
            for dc in (-1,0,1):
                out += cells.get(((a+da)%n,(b+db)%n,(c+dc)%n),[])
    return [j for j in out if j!=i]

def _pe(i,pos,R,P,eps,nbrs,L):
    e=0.0
    for j in nbrs:
        dr=pos[j]-pos[i]; dr-=L*np.round(dr/L); r=np.sqrt(dr@dr)
        if r<R_WELL: e+=pair_energy(dr,r,R[i],R[j],P,eps)
    return e

# ---- order parameters -------------------------------------------------------
def quality(pos,R,P,eps,L):
    """(largest_cluster, well_ordered_fraction, mean_coordination).
    A bond requires energy < -0.4*eps (a geometrically correct interface, not a
    grazing contact). well_ordered_fraction = fraction with coordination >= 3,
    i.e. subunits locked into a shell fragment rather than a dangling edge."""
    N=len(pos); cells,ncell,cs=build_cells(pos,L,R_WELL)
    coord=np.zeros(N,int); parent=list(range(N))
    def find(x):
        while parent[x]!=x: parent[x]=parent[parent[x]]; x=parent[x]
        return x
    for i in range(N):
        for j in neighbors(i,pos,cells,ncell,cs,L):
            if j<=i: continue
            dr=pos[j]-pos[i]; dr-=L*np.round(dr/L); r=np.sqrt(dr@dr)
            if r<R_WELL and pair_energy(dr,r,R[i],R[j],P,eps)<-0.4*eps:
                coord[i]+=1; coord[j]+=1
                a,b=find(i),find(j)
                if a!=b: parent[a]=b
    sizes=Counter(find(i) for i in range(N))
    return max(sizes.values()), float((coord>=3).mean()), float(coord.mean())

# ---- MC driver --------------------------------------------------------------
def run_mc(N, phi, eps, patches="curved5", sweeps=1200, seed=0,
           trans_step=0.35, d_rot=0.45, record_every=100):
    """NVT Metropolis MC of N patchy subunits at packing fraction phi and bond
    energy eps (units of kT). Returns (trajectory, pos, R, L) where trajectory
    rows are (sweep, largest_cluster, well_ordered_fraction, mean_coordination)."""
    rng = np.random.default_rng(seed)
    P = make_patches(patches)
    L = (N*(np.pi/6)*SIGMA**3 / phi)**(1/3)
    pos = rng.random((N,3))*L
    R = np.stack([rand_rot(rng) for _ in range(N)])
    traj=[]
    for sw in range(sweeps):
        cells,ncell,cs = build_cells(pos,L,R_WELL)
        for i in rng.permutation(N):
            nbrs = neighbors(i,pos,cells,ncell,cs,L)
            e_old=_pe(i,pos,R,P,eps,nbrs,L)
            old_p=pos[i].copy(); old_R=R[i].copy()
            if rng.random()<0.5:
                pos[i]=(pos[i]+(rng.random(3)-0.5)*trans_step)%L
            else:
                R[i]=small_rot(rng,d_rot)@R[i]
            e_new=_pe(i,pos,R,P,eps,nbrs,L)
            de=e_new-e_old
            if np.isnan(de): accept=(e_new!=np.inf)          # overlap->overlap
            else:            accept=(de<=0) or (rng.random()<np.exp(-min(de,700)))
            if not accept: pos[i]=old_p; R[i]=old_R
        if sw%record_every==0 or sw==sweeps-1:
            traj.append((sw,)+quality(pos,R,P,eps,L))
    return np.array(traj), pos, R, L

# ---- demonstration: the kinetic trap (concentration scan) -------------------
if __name__ == "__main__":
    print("Concentration scan (Mode A prediction, no wet-lab data):")
    print(" phi   well-ordered   mean-coord   largest")
    for phi in [0.03,0.06,0.10,0.15,0.22,0.30]:
        q=[run_mc(N=90,phi=phi,eps=11.0,sweeps=1200,seed=s,record_every=1200)[0][-1]
           for s in range(3)]
        q=np.array(q)
        print(f" {phi:.2f}   {q[:,2].mean():.3f}         {q[:,3].mean():.2f}"
              f"         {q[:,1].mean():.0f}")
    print("\nPrediction: quality is non-monotonic in concentration - the kinetic")
    print("trap, reproduced independently from the Module 4 mean-field ODEs.")


In [ ]:
%%writefile M4_bridge.py
"""
M4 multiscale bridge — CG Monte Carlo -> ODE cascade -> benchtop data.
BruSafe / CAPSID, iGEM 2026.

The Zlotnick-Hagan rate-equation cascade concentrates its assembly physics in one
object: c(j), the number of subunit-subunit contacts formed when a cluster of j-1
grows to j. Reverse rates follow from it by detailed balance, so c(j) sets the
nucleation barrier, the critical nucleus, and the peaked yield curve.

The published engine hard-codes c(j) as a 1/2/3 step function. This module MEASURES
it from coarse-grained configurations instead, then propagates it into the ODE and
into two benchtop observables (plate-reader turbidity, TEM size distribution).

Requires: brusafe_cg_assembly.py (CG engine), brusafe_synthesis.py (ODE cascade).
"""
import numpy as np, pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit, least_squares

N_CAP = 90          # dimers per T=3 MS2 shell
RT    = 0.593      # kcal/mol at 298 K
BTOT  = N_CAP*5/2   # bonds in the closed shell (5-coordinate lattice)

# ---------------------------------------------------------------- 1. extraction
def cluster_stats(pos, R, P, eps, L, gcg):
    """Per-cluster (size, bond_count) from one CG configuration.
    Bond criterion is the ENGINE's own: pair energy < -0.4*eps. Sharing one
    definition of 'contact' across scales is what makes the bridge meaningful."""
    build_cells, neighbors = gcg["build_cells"], gcg["neighbors"]
    pair_energy, R_WELL = gcg["pair_energy"], gcg["R_WELL"]
    N = len(pos); cells, ncell, cs = build_cells(pos, L, R_WELL)
    parent = list(range(N))
    def find(x):
        while parent[x] != x: parent[x] = parent[parent[x]]; x = parent[x]
        return x
    bonds = []
    for i in range(N):
        for j in neighbors(i, pos, cells, ncell, cs, L):
            if j <= i: continue
            dr = pos[j]-pos[i]; dr -= L*np.round(dr/L); r = float(np.sqrt(dr@dr))
            if r < R_WELL and pair_energy(dr, r, R[i], R[j], P, eps) < -0.4*eps:
                bonds.append((i, j))
                a, b = find(i), find(j)
                if a != b: parent[a] = b
    lab = [find(i) for i in range(N)]
    from collections import Counter
    size = Counter(lab)
    bcnt = Counter()
    for i, j in bonds: bcnt[find(i)] += 1
    return [(size[k], bcnt[k]) for k in size]

# --------------------------------------------------------------- 2. contact law
S1 = np.sqrt(1.0*(1.0-1.0/N_CAP))
def b_cap(j, a):
    """Bonds in a compact cap of j subunits on the shell lattice.
    2.5*(j-1) is the closed-shell bulk term; the sqrt term is the unsatisfied
    open perimeter, vanishing as the cap closes at j=N_CAP. b(1)=0 exactly."""
    return 2.5*(j-1.0) - a*(np.sqrt(np.clip(j*(1.0-j/N_CAP), 0, None)) - S1)

def fit_contact_law(sizes, bonds, weights=None):
    """Fit the single shape parameter a to the CG bond census."""
    sig = None if weights is None else 1.0/np.asarray(weights)
    popt, pcov = curve_fit(b_cap, np.asarray(sizes, float), np.asarray(bonds, float),
                           p0=[3.0], sigma=sig, absolute_sigma=False)
    return float(popt[0]), float(np.sqrt(pcov[0, 0]))

def ladder_from_a(a):
    """Marginal contacts c(j), j = 1..N_CAP, from the fitted cap law."""
    jg = np.arange(1, N_CAP+1, dtype=float)
    b = np.maximum.accumulate(np.clip(b_cap(jg, a), 0.0, BTOT))
    return jg, b, np.diff(np.concatenate([np.zeros(1), b]))

def critical_nucleus(a, g_assoc, g_contact):
    """n_c is the argmax of dG(j) = j*g_assoc + b(j)*g_contact -- EMERGENT from
    the measured geometry plus two energies, not an input parameter."""
    jg, b, _ = ladder_from_a(a)
    prof = jg*g_assoc + b*g_contact
    return int(jg[np.argmax(prof)]), float(prof.max())

# ------------------------------------------------------------------ 3. cascade
def build_rates_from_ladder(k_a, g_assoc, g_contact, c_ladder):
    """Detailed balance, identical construction to the published engine, but the
    contact ladder is supplied rather than hard-coded."""
    f = np.full(N_CAP, float(k_a)); b = np.zeros(N_CAP)
    for j in range(1, N_CAP):
        b[j] = k_a*np.exp((g_assoc + c_ladder[j]*g_contact)/RT)
    return f, b

def cascade(D0, f, b, t_max=240.0, nt=600):
    x0 = np.zeros(N_CAP); x0[0] = D0
    def rhs(t, x):
        dx = np.zeros_like(x)
        fl = f[:N_CAP-1]*x[0]*x[:N_CAP-1] - b[1:]*x[1:]
        dx[:N_CAP-1] -= fl; dx[1:] += fl; dx[0] -= fl.sum()
        return dx
    s = solve_ivp(rhs, (0, t_max), x0, t_eval=np.linspace(0, t_max, nt),
                  method="LSODA", rtol=1e-8, atol=1e-12)
    return s.t, s.y

def observables(y):
    x = y[:, -1]; j = np.arange(1, N_CAP+1); m = x*j; tot = m.sum()
    return dict(capsid=float(m[-1]/tot), free=float(m[0]/tot),
                inter=float(m[1:-1].sum()/tot))

# -------------------------------------------------------------- 4. observables
def turbidity(D0, ga, gc, c_ladder, ka=3.0, t_max=240.0, nt=241):
    """Plate-reader signal. For particles small vs lambda, Rayleigh scattering per
    species scales as (mass)^2, so tau ~ sum_j x_j*j^2. This is NOT capsid fraction:
    partial shells scatter too, which is exactly why turbidity alone cannot
    distinguish productive assembly from kinetic trapping -- you need TEM."""
    f, b = build_rates_from_ladder(ka, ga, gc, c_ladder)
    t, y = cascade(D0, f, b, t_max=t_max, nt=nt)
    j = np.arange(1, N_CAP+1)[:, None]
    tau = (y*(j**2)).sum(axis=0)
    # normalise by the signal of complete conversion to closed capsids:
    # x_N = D0/N, so tau_max = (D0/N)*N^2 = D0*N. Dividing by D0*N^2 instead
    # squashes the whole curve by a factor N and makes the axis unreadable.
    return t, tau/(D0*N_CAP)

def size_distribution(D0, ga, gc, c_ladder, ka=3.0, t=240.0):
    """Number- and mass-weighted size distributions: what a TEM grid shows."""
    f, b = build_rates_from_ladder(ka, ga, gc, c_ladder)
    _, y = cascade(D0, f, b, t_max=t, nt=120)
    x = y[:, -1]; j = np.arange(1, N_CAP+1); m = x*j
    return j, x/x.sum(), m/m.sum()

# --------------------------------------------------------------- 5. correction
def yield_curve(ga, gc, c_ladder, dgrid, ka=3.0):
    f, b = build_rates_from_ladder(ka, ga, gc, c_ladder)
    return np.array([observables(cascade(d, f, b, nt=200)[1])["capsid"] for d in dgrid])

def optimum(ga, gc, c_ladder, dgrid, ka=3.0):
    y = yield_curve(ga, gc, c_ladder, dgrid, ka); k = int(np.argmax(y))
    return float(dgrid[k]), float(y[k]), bool(k == len(dgrid)-1)

def invert_energies(D_meas, Y_meas, c_ladder, dgrid, x0=(4.0, -4.0)):
    """CORRECTIVE step: two observables (optimum position, optimum height) -> two
    energies, with the CG-measured geometry held FIXED. Geometry comes from
    simulation, energetics from the bench."""
    def resid(p):
        d, yv, _ = optimum(p[0], p[1], c_ladder, dgrid)
        return [np.log(d/D_meas), yv - Y_meas]
    s = least_squares(resid, x0=list(x0), bounds=([1.0, -6.0], [8.0, -2.5]),
                      diff_step=0.08, xtol=1e-3, ftol=1e-3, max_nfev=40)
    return float(s.x[0]), float(s.x[1]), np.asarray(s.fun)

# ------------------------------------------------------------------ provenance
FITTED = dict(a=6.754255651074951, a_sd=0.3745267255362961, j_measured_max=29,
              n_clusters=2547, eps_sampled=(6.0, 7.0, 8.0))


## 1. The measured cap-law parameter

`FITTED` records the published fit and its provenance: the bond census of 2547 CG clusters sampled at three interface strengths. Sizes above 29 dimers were not observed in the CG box, so the cap law is *extrapolated* beyond there — a stated limit, not a hidden one.

In [ ]:
import M4_bridge as BR
import pandas as pd

print(BR.FITTED)
a = BR.FITTED['a']; a_sd = BR.FITTED['a_sd']
jg, b, c = BR.ladder_from_a(a)
print(f'\ncap-law shape parameter a = {a:.3f} +/- {a_sd:.3f}')
print('contact ladder c(j) at j = 2, 5, 10, 45, 89, 90:',
      [round(float(c[k-1]), 2) for k in (2, 5, 10, 45, 89, 90)])

### Optional — re-measure the ladder from a fresh CG run

Slow (a few minutes). Skip it to use the published fit.

In [ ]:
RERUN_CG = False   # set to True to re-measure the ladder

if RERUN_CG:
    import brusafe_cg_assembly as L3
    gcg = dict(build_cells=L3.build_cells, neighbors=L3.neighbors,
               pair_energy=L3.pair_energy, R_WELL=L3.R_WELL)
    Pp = L3.make_patches('curved5')
    sizes, bonds = [], []
    for eps in (6.0, 7.0, 8.0):
        for seed in range(2):
            _, pos, Rm, L = L3.run_mc(N=90, phi=0.15, eps=eps, sweeps=400,
                                      seed=seed, record_every=400)
            for s, bn in BR.cluster_stats(pos, Rm, Pp, eps, L, gcg):
                sizes.append(s); bonds.append(bn)
    a_new, sd_new = BR.fit_contact_law(sizes, bonds)
    print(f'{len(sizes)} clusters -> a = {a_new:.3f} +/- {sd_new:.3f}')
    a = a_new

## 2. The critical nucleus becomes an *output*

In L1, $n_c$ is a parameter you set. Here it is the argmax of $\Delta G(j) = j\,g_{assoc} + b(j)\,g_{contact}$ — the measured geometry plus two energies determine it.

In [ ]:
c_ladder = c
g_assoc, g_contact = 6.0, -4.0
nc, Gmax = BR.critical_nucleus(a, g_assoc, g_contact)
print(f'emergent critical nucleus n_c = {nc} dimers (barrier {Gmax:.2f} kcal/mol)')

prof = jg*g_assoc + b*g_contact
fig, ax = plt.subplots(figsize=(5.8, 3.8))
ax.plot(jg, prof, lw=2, color='#0b6e5a')
ax.axvline(nc, ls=':', color='#b3402f')
ax.annotate(f'$n_c$ = {nc}', xy=(nc, Gmax), xytext=(nc+8, Gmax*0.85),
            fontsize=9, arrowprops=dict(arrowstyle='->', color='0.4'))
ax.set_xlabel('cluster size $j$ (dimers)'); ax.set_ylabel(r'$\Delta G(j)$ (kcal/mol)')
ax.set_title('Free-energy profile from the measured cap law')
fig.tight_layout(); fig.savefig('M4_bridge_nucleus.png', dpi=200)

## 3. Corrective step: inverting measured data into energies

The measured position and height of the yield peak are entered below. The solver returns the $(g_{assoc}, g_{contact})$ pair that reproduces them, with the geometry measured by CG simulation held fixed.

Two observables determine two unknowns, which is why the problem is well posed and why a third energy could not be fitted from the same curve.

In [ ]:
D_MEAS = 2.5     # measured optimum position (uM dimer); example value
Y_MEAS = 0.60    # measured capsid mass fraction at that optimum; example value

dgrid = np.geomspace(0.3, 12, 26)
ga_fit, gc_fit, resid = BR.invert_energies(D_MEAS, Y_MEAS, c_ladder, dgrid)
print(f'g_assoc   = {ga_fit:+.2f} kcal/mol')
print(f'g_contact = {gc_fit:+.2f} kcal/mol')
print(f'residuals (log position, height) = {resid}')
lo, hi = -4.4, -2.9   # Ceres 2002 interval for a related capsid
print('inside the -2.9 to -4.4 kcal/mol literature interval:', lo <= gc_fit <= hi)

In [ ]:
d0, y0, _ = BR.optimum(g_assoc, g_contact, c_ladder, dgrid)
yc_prior = BR.yield_curve(g_assoc, g_contact, c_ladder, dgrid)
yc_post  = BR.yield_curve(ga_fit,  gc_fit,   c_ladder, dgrid)

fig, ax = plt.subplots(figsize=(6.2, 4.1))
ax.plot(dgrid, yc_prior*100, '--', lw=1.8, color='0.55', label='prior (literature energies)')
ax.plot(dgrid, yc_post*100,  '-',  lw=2.2, color='#0b6e5a', label='posterior (fitted to your data)')
ax.plot([D_MEAS], [Y_MEAS*100], 'o', ms=9, color='#b3402f', zorder=5, label='your measurement')
ax.set_xscale('log'); ax.set_xlabel(r'$[D]_0$ ($\mu$M)')
ax.set_ylabel('closed capsid (% of coat mass)')
ax.set_title('Model before and after the bench')
ax.legend(frameon=False, fontsize=9); fig.tight_layout()
fig.savefig('M4_bridge_correction.png', dpi=200)
print(f'prior optimum {d0:.2f} uM at {y0:.2f}; measurement {D_MEAS:.2f} uM at {Y_MEAS:.2f}')

## 4. Two benchtop observables

**Turbidity** is what a plate reader gives you. For particles small relative to the wavelength, Rayleigh scattering per species goes as mass², so the signal is $\tau \sim \sum_j x_j j^2$ — *partial shells scatter too*. That is exactly why turbidity alone cannot separate productive assembly from kinetic trapping, and why TEM is not optional.

**Size distribution** is what a TEM grid gives you.

In [ ]:
# plotted with the POSTERIOR energies from the inversion above. With the
# literature prior the CG-measured ladder assembles nothing at all (see the
# dashed curve in the next figure) -- so a prior-energy TEM prediction would
# just be a flat line at the monomer peak.
fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.9))

for D0, col in zip([1.0, 2.5, 5.0], ['#c8863c', '#0b6e5a', '#b3402f']):
    tt, tau = BR.turbidity(D0, ga_fit, gc_fit, c_ladder, ka=3.0)
    axes[0].plot(tt, tau, lw=2, color=col, label=fr'$[D]_0$={D0} $\mu$M')
axes[0].set_xlabel('time (min)')
axes[0].set_ylabel('turbidity / turbidity of full conversion')
axes[0].set_title('Plate-reader signal'); axes[0].legend(frameon=False, fontsize=9)

for D0, col in zip([1.0, 2.5, 5.0], ['#c8863c', '#0b6e5a', '#b3402f']):
    jj, xn, xm = BR.size_distribution(D0, ga_fit, gc_fit, c_ladder, ka=3.0)
    axes[1].semilogy(jj, np.clip(xm, 1e-6, None), lw=1.8, color=col,
                     label=fr'$[D]_0$={D0} $\mu$M')
axes[1].set_ylim(1e-4, 1.5)
axes[1].set_xlabel('cluster size (dimers)'); axes[1].set_ylabel('mass fraction')
axes[1].set_title('What a TEM grid should show'); axes[1].legend(frameon=False, fontsize=9)
fig.tight_layout(); fig.savefig('M4_bridge_observables.png', dpi=200)

for D0 in (1.0, 2.5, 5.0):
    jj, xn, xm = BR.size_distribution(D0, ga_fit, gc_fit, c_ladder, ka=3.0)
    print(f'[D]0={D0:4.1f} uM  monomer {xm[0]:.2f}  '
          f'partial {xm[1:-1].sum():.2f}  closed capsid {xm[-1]:.2f}')

---
**Outputs:** `M4_bridge_nucleus.png`, `M4_bridge_observables.png`, `M4_bridge_correction.png`.

The last figure shows the model changing in response to the measurements, which is the purpose of the correction step.